# CloudWatch-Style Intrusion Detection Assignment\n
\n
Paired with `src/train.py` for a full pipeline run.

## Part 1 - Defining the Problem\n
The goal of this project is to detect malicious or suspicious network behavior from log data.

- **Attacker behavior to detect:** suspicious/malicious network interactions (scanning, probing, brute-force, suspicious request patterns).\n
- **False positive cost:** analyst fatigue, wasted triage time, possible blocking of legitimate users.\n
- **False negative cost:** missed intrusion, potential data loss and persistence.\n
- **Why NN may help (or not):** NNs can capture nonlinear interactions; tree/linear baselines can still be strong for tabular data.

In [ ]:
from pathlib import Path\n
import sys\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
\n
ROOT = Path.cwd()\n
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():\n
    ROOT = ROOT.parent\n
sys.path.append(str(ROOT))\n
\n
from src.train import (\n
    infer_target_column,\n
    build_binary_target,\n
    build_preprocessor,\n
    densify_if_needed,\n
    build_nn,\n
    train_torch_nn,\n
    predict_torch_probs,\n
    compute_metrics,\n
    export_failure_cases,\n
)\n
from sklearn.model_selection import train_test_split\n
from sklearn.linear_model import LogisticRegression\n
\n
RANDOM_STATE = 42\n

In [ ]:
# Set your dataset path and (optionally) target column name\n
DATA_PATH = ROOT / 'data' / 'your_dataset.csv'\n
TARGET_COLUMN = None  # e.g., 'label'\n
POSITIVE_LABELS = None  # e.g., {'attack', 'malicious'}\n
MAX_CAT_CARDINALITY = 100\n
\n
df = pd.read_csv(DATA_PATH)\n
df.columns = [str(c).strip() for c in df.columns]\n
df = df.drop_duplicates().reset_index(drop=True)\n
\n
target_col = infer_target_column(df, TARGET_COLUMN)\n
y, target_mapping_note = build_binary_target(df[target_col], POSITIVE_LABELS)\n
X = df.drop(columns=[target_col]).copy()\n
\n
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]\n
if constant_cols:\n
    X = X.drop(columns=constant_cols)\n
\n
print(f'Target column: {target_col}')\n
print(f'Target mapping: {target_mapping_note}')\n
print('Class distribution:')\n
print(y.value_counts(normalize=True))\n

## Part 2 - Data Preparation & Feature Engineering\n
\n
This section performs split, missing-value handling, categorical encoding, and numeric normalization.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(\n
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE\n
)\n
X_train, X_val, y_train, y_val = train_test_split(\n
    X_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=RANDOM_STATE\n
)\n
\n
preprocessor, numeric_cols, categorical_cols, dropped_high_card = build_preprocessor(\n
    X_train, MAX_CAT_CARDINALITY\n
)\n
\n
X_train_t = densify_if_needed(preprocessor.fit_transform(X_train))\n
X_val_t = densify_if_needed(preprocessor.transform(X_val))\n
X_test_t = densify_if_needed(preprocessor.transform(X_test))\n
\n
print(f'Train shape after preprocessing: {X_train_t.shape}')\n
print(f'Numeric columns: {len(numeric_cols)}')\n
print(f'Categorical columns (encoded): {len(categorical_cols)}')\n
print(f'High-cardinality categorical columns dropped: {dropped_high_card}')\n

## Part 3 - Neural Network Classifier\n
\n
Architecture constraints satisfied:\n
- Fully connected feedforward\n
- 3 hidden layers\n
- ReLU hidden activations\n
- Sigmoid output

In [ ]:
import torch\n
from sklearn.utils.class_weight import compute_class_weight\n
\n
torch.manual_seed(RANDOM_STATE)\n
\n
class_vals = np.array([0, 1], dtype=np.int64)\n
cw = compute_class_weight(class_weight='balanced', classes=class_vals, y=np.asarray(y_train, dtype=np.int64))\n
class_weight = {0: float(cw[0]), 1: float(cw[1])}\n
\n
nn_model = build_nn(X_train_t.shape[1])\n
history = train_torch_nn(\n
    model=nn_model,\n
    X_train=np.asarray(X_train_t, dtype=np.float32),\n
    y_train=np.asarray(y_train, dtype=np.float32),\n
    X_val=np.asarray(X_val_t, dtype=np.float32),\n
    y_val=np.asarray(y_val, dtype=np.float32),\n
    class_weights=class_weight,\n
    epochs=40,\n
    batch_size=256,\n
    patience=5,\n
)\n

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n
axes[0].plot(history['loss'], label='train_loss')\n
axes[0].plot(history['val_loss'], label='val_loss')\n
axes[0].set_title('Loss')\n
axes[0].legend()\n
\n
axes[1].plot(history['accuracy'], label='train_acc')\n
axes[1].plot(history['val_accuracy'], label='val_acc')\n
axes[1].set_title('Accuracy')\n
axes[1].legend()\n
plt.tight_layout()\n
plt.show()\n

In [ ]:
nn_probs = predict_torch_probs(nn_model, np.asarray(X_test_t, dtype=np.float32))\n
nn_preds = (nn_probs >= 0.5).astype(int)\n
nn_metrics = compute_metrics(np.asarray(y_test), nn_preds)\n
nn_metrics\n

## Part 4 - Baseline Comparison\n
\n
Train a non-NN model and compare Accuracy, Precision, Recall, F1, and FP/FN.

In [ ]:
baseline = LogisticRegression(max_iter=3000, class_weight='balanced')\n
baseline.fit(X_train_t, np.asarray(y_train))\n
\n
base_probs = baseline.predict_proba(X_test_t)[:, 1]\n
base_preds = (base_probs >= 0.5).astype(int)\n
base_metrics = compute_metrics(np.asarray(y_test), base_preds)\n
\n
comparison = pd.DataFrame([\n
    {'model': 'NeuralNetwork', **nn_metrics},\n
    {'model': 'LogisticRegression', **base_metrics},\n
])\n
comparison\n

## Part 5 - Failure Case Analysis\n
\n
This exports up to 5 NN misclassified examples with a starter failure-type tag.

In [ ]:
fail_path = ROOT / 'outputs' / 'failure_cases_notebook.csv'\n
fail_path.parent.mkdir(parents=True, exist_ok=True)\n
mis_total = export_failure_cases(\n
    X_test_raw=X_test,\n
    y_true=np.asarray(y_test),\n
    y_pred=nn_preds,\n
    y_prob=nn_probs,\n
    output_csv=fail_path,\n
    max_rows=5,\n
)\n
print(f'Total NN misclassifications: {mis_total}')\n
pd.read_csv(fail_path).head(5)\n

## Part 6 - Thought Exercise\n
- Should this model operate autonomously? No for high-impact actions)\n
- How should analysts interact with it? Prioritized alerts + explanations + feedback loop)\n
- Ethical risks of false accusations? legitimate users blocked/investigated unfairly\n
- How could bias manifest? Over-flagging certain geographies, ASNs, device types from skewed data

## Optional: One-command run\n
\n
You can also run everything from script mode:\n
\n
```bash\n
python src/train.py --data data/dionaeaClean2.csv --target protocol --positive-labels mssqld,httpd,mysqld,pptpd,ftpd,mongod,epmapper,mqttd
```\n
\n
Artifacts are saved to `outputs/`.